# 01. Основы визуализации данных

Кейс: **«РегионМаркет»**.

## Результат
После ноутбука вы сможете выбрать базовый тип графика, подготовить данные через `groupby`, построить график через объектный API Matplotlib и сделать оси читаемыми.

## 1. Аналитический вопрос важнее библиотеки

Сначала формулируем вопрос, затем выбираем график.

| Вопрос | Тип |
|---|---|
| Что происходит во времени? | line |
| Какая категория лидирует? | bar |
| Как распределены значения? | histogram |
| Есть ли выбросы? | boxplot |
| Есть ли связь двух числовых признаков? | scatter |

In [ ]:
from pathlib import Path

def find_project_root() -> Path:
    """Ищет корень учебного комплекта по наличию подготовленного датасета."""
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        target = candidate / "data" / "viz.csv"
        if target.exists():
            return candidate
    raise FileNotFoundError("Не найден data/viz.csv")

PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "viz.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("Data:", DATA_PATH)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import AutoMinorLocator, FuncFormatter

print("pandas:", pd.__version__)
import matplotlib
print("matplotlib:", matplotlib.__version__)

In [ ]:
df = pd.read_csv(DATA_PATH)
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
print("Shape:", df.shape)
display(df.head())

## 2. Минимальная проверка данных перед графиком

Нельзя строить выводы, не проверив хотя бы размер, тип даты, пропуски ключевой метрики и дубликаты.

In [ ]:
print("Период:", df["order_date"].min().date(), "—", df["order_date"].max().date())
print("Пропуски revenue:", df["revenue"].isna().sum())
print("Полные дубликаты:", df.duplicated().sum())
print("Уникальные заказы:", df["order_id"].nunique())

## 3. Динамика выручки

**Вопрос:** как менялась выручка по месяцам?

Сначала агрегируем данные до уровня месяца.

In [ ]:
monthly = (
    df.assign(month=df["order_date"].dt.to_period("M").dt.to_timestamp())
      .groupby("month", as_index=False)
      .agg(revenue=("revenue", "sum"), profit=("profit", "sum"), orders=("order_id", "nunique"))
)
display(monthly)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(monthly["month"], monthly["revenue"], marker="o")
ax.set_title("Динамика выручки по месяцам")
ax.set_xlabel("Месяц")
ax.set_ylabel("Выручка, руб.")
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m.%Y"))
ax.xaxis.set_minor_locator(mdates.WeekdayLocator(byweekday=mdates.MO, interval=2))
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x/1_000_000:.1f} млн"))
ax.yaxis.set_minor_locator(AutoMinorLocator(2))
ax.grid(True, which="major")
ax.grid(True, which="minor", linewidth=0.35, alpha=0.5)
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "01_monthly_revenue.png", dpi=150, bbox_inches="tight")
plt.show()

### Контрольная точка 1

Ответьте письменно:

1. В каком месяце выручка максимальна?
2. Видим ли мы причину роста на самом графике?
3. Какое дополнительное разбиение поможет объяснить динамику?

## 4. Сравнение категорий

**Вопрос:** какие категории формируют основную выручку?

In [ ]:
category_revenue = (
    df.groupby("category", as_index=False)
      .agg(revenue=("revenue", "sum"))
      .sort_values("revenue", ascending=True)
)
display(category_revenue)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(category_revenue["category"], category_revenue["revenue"])
ax.set_title("Выручка по категориям")
ax.set_xlabel("Выручка, руб.")
ax.set_ylabel("Категория")
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x/1_000_000:.1f} млн"))
ax.xaxis.set_minor_locator(AutoMinorLocator(2))
ax.grid(True, axis="x", which="major")
ax.bar_label(bars, labels=[f"{v/1_000_000:.1f} млн" for v in category_revenue["revenue"]], padding=3)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "02_category_revenue.png", dpi=150, bbox_inches="tight")
plt.show()

### Контрольная точка 2

Почему для сравнения категорий горизонтальная диаграмма часто удобнее вертикальной, когда подписи длинные?

## Самопроверка

- [ ] я могу объяснить, почему выбран line/bar;
- [ ] дата отсортирована и ось читается;
- [ ] я понимаю, на каком уровне агрегированы данные;
- [ ] у графика есть содержательный вывод, а не только заголовок.